In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import pandas as pd
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, max_error, mean_absolute_percentage_error
import threading
from datetime import datetime
import json
import os
import warnings
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List, Any
from enum import Enum
import logging
import traceback
import random

warnings.filterwarnings('ignore')

# CatBoost import
from catboost import CatBoostRegressor

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class Theme(Enum):
    """Application themes"""
    LIGHT = "light"
    DARK = "dark"
    PROFESSIONAL = "professional"
    MODERN = "modern"


@dataclass
class ColorPalette:
    """Enhanced color scheme management"""
    def __init__(self, theme: Theme = Theme.PROFESSIONAL):
        self.theme = theme
        self.update_colors()
    
    def update_colors(self):
        if self.theme == Theme.LIGHT:
            self.bg_main = '#F8F9FA'
            self.card = '#FFFFFF'
            self.text_primary = '#2C3E50'
            self.text_secondary = '#7F8C8D'
            self.border = '#E1E8ED'
            self.primary = '#3498DB'
            self.secondary = '#9B59B6'
            self.success = '#27AE60'
            self.warning = '#F39C12'
            self.danger = '#E74C3C'
            self.accent = '#E67E22'
            self.gradient_start = '#3498DB'
            self.gradient_end = '#2980B9'
            self.subtitle_color = '#34495E'
        elif self.theme == Theme.DARK:
            self.bg_main = '#1a1a2e'
            self.card = '#16213e'
            self.text_primary = '#EEEEEE'
            self.text_secondary = '#AAAAAA'
            self.border = '#0f3460'
            self.primary = '#00b4d8'
            self.secondary = '#9B59B6'
            self.success = '#00f5d4'
            self.warning = '#fca311'
            self.danger = '#ef476f'
            self.accent = '#ff006e'
            self.gradient_start = '#00b4d8'
            self.gradient_end = '#0077b6'
            self.subtitle_color = '#CCCCCC'
        elif self.theme == Theme.PROFESSIONAL:
            self.bg_main = '#F4F6F9'
            self.card = '#FFFFFF'
            self.text_primary = '#1F2937'
            self.text_secondary = '#6B7280'
            self.border = '#E5E7EB'
            self.primary = '#2563EB'
            self.secondary = '#7C3AED'
            self.success = '#059669'
            self.warning = '#D97706'
            self.danger = '#DC2626'
            self.accent = '#EA580C'
            self.gradient_start = '#2563EB'
            self.gradient_end = '#7C3AED'
            self.subtitle_color = '#4B5563'
        else:  # MODERN
            self.bg_main = '#0f0c29'
            self.card = '#1a1a3e'
            self.text_primary = '#ffffff'
            self.text_secondary = '#b8b8ff'
            self.border = '#2d2d5e'
            self.primary = '#667eea'
            self.secondary = '#764ba2'
            self.success = '#00d4aa'
            self.warning = '#ffd93d'
            self.danger = '#ff6b6b'
            self.accent = '#ff6b6b'
            self.gradient_start = '#667eea'
            self.gradient_end = '#764ba2'
            self.subtitle_color = '#c4c4f0'
        
        self.plot_colors = {
            'low_strength': self.danger,
            'medium_strength': self.warning,
            'high_strength': self.success,
            'comparison': [self.success, self.primary, self.warning, self.danger, self.secondary],
            'gradient': plt.cm.RdYlGn
        }


class MOWCA:
    """Modified Water Cycle Algorithm for hyperparameter optimization"""
    
    def __init__(self, objective_func, n_streams=20, n_seas=4, n_rivers=4, 
                 max_iter=50, lb=None, ub=None):
        """
        MOWCA Optimizer
        
        Parameters:
        - objective_func: function to minimize/maximize
        - n_streams: number of streams (population size)
        - n_seas: number of seas
        - n_rivers: number of rivers
        - max_iter: maximum iterations
        - lb: lower bounds for parameters
        - ub: upper bounds for parameters
        """
        self.objective_func = objective_func
        self.n_streams = n_streams
        self.n_seas = n_seas
        self.n_rivers = n_rivers
        self.max_iter = max_iter
        self.lb = np.array(lb) if lb is not None else None
        self.ub = np.array(ub) if ub is not None else None
        
        # Best solution tracking
        self.best_solution = None
        self.best_fitness = float('inf')
        self.convergence_curve = []
        
    def optimize(self):
        """Run MOWCA optimization"""
        # Initialize population
        population = self._initialize_population()
        fitness = np.array([self.objective_func(ind) for ind in population])
        
        # Sort by fitness (lower is better)
        sorted_idx = np.argsort(fitness)
        fitness = fitness[sorted_idx]
        population = population[sorted_idx]
        
        # Assign sea and rivers
        seas = population[:self.n_seas]
        rivers = population[self.n_seas:self.n_seas+self.n_rivers]
        streams = population[self.n_seas+self.n_rivers:]
        
        seas_fitness = fitness[:self.n_seas]
        rivers_fitness = fitness[self.n_seas:self.n_seas+self.n_rivers]
        
        # Update best solution
        if seas_fitness[0] < self.best_fitness:
            self.best_fitness = seas_fitness[0]
            self.best_solution = seas[0].copy()
        
        # Main optimization loop
        for iteration in range(self.max_iter):
            # Calculate flow intensities
            total_fitness = np.sum(1/(seas_fitness + 1e-10)) + np.sum(1/(rivers_fitness + 1e-10))
            
            # Probability of sea
            sea_prob = (1/(seas_fitness[0] + 1e-10)) / total_fitness
            
            # Update streams flowing to rivers
            for i in range(len(streams)):
                # Choose destination (river or sea)
                if np.random.random() < sea_prob:
                    destination = seas[0]  # Flow to sea
                else:
                    # Choose a river
                    river_idx = np.random.choice(len(rivers))
                    destination = rivers[river_idx]
                
                # Update stream position
                C = 2 * (1 - iteration/self.max_iter)  # Decreasing coefficient
                streams[i] = streams[i] + C * np.random.random() * (destination - streams[i])
                
                # Boundary check
                streams[i] = np.clip(streams[i], self.lb, self.ub)
            
            # Update rivers flowing to sea
            for i in range(len(rivers)):
                # Evaporation condition
                if np.random.random() < 0.1 * (1 - iteration/self.max_iter):
                    # Evaporation and raining process
                    rivers[i] = self.lb + np.random.random(len(self.lb)) * (self.ub - self.lb)
                else:
                    # Flow to sea
                    rivers[i] = rivers[i] + np.random.random() * (seas[0] - rivers[i])
                    rivers[i] = np.clip(rivers[i], self.lb, self.ub)
            
            # Update sea position
            seas[0] = seas[0] + np.random.random() * (seas[0] - np.mean(rivers, axis=0))
            seas[0] = np.clip(seas[0], self.lb, self.ub)
            
            # Evaluate new positions
            new_fitness_streams = np.array([self.objective_func(s) for s in streams])
            new_fitness_rivers = np.array([self.objective_func(r) for r in rivers])
            new_fitness_seas = np.array([self.objective_func(s) for s in seas])
            
            # Combine all
            all_pop = np.vstack([seas, rivers, streams])
            all_fitness = np.concatenate([new_fitness_seas, new_fitness_rivers, new_fitness_streams])
            
            # Sort
            sorted_idx = np.argsort(all_fitness)
            all_fitness = all_fitness[sorted_idx]
            all_pop = all_pop[sorted_idx]
            
            # Update seas and rivers
            seas = all_pop[:self.n_seas]
            rivers = all_pop[self.n_seas:self.n_seas+self.n_rivers]
            streams = all_pop[self.n_seas+self.n_rivers:]
            
            seas_fitness = all_fitness[:self.n_seas]
            rivers_fitness = all_fitness[self.n_seas:self.n_seas+self.n_rivers]
            
            # Update best solution
            if seas_fitness[0] < self.best_fitness:
                self.best_fitness = seas_fitness[0]
                self.best_solution = seas[0].copy()
            
            self.convergence_curve.append(self.best_fitness)
            
            logger.info(f"MOWCA Iteration {iteration+1}/{self.max_iter}, Best Fitness: {self.best_fitness:.6f}")
        
        return self.best_solution, self.best_fitness, self.convergence_curve
    
    def _initialize_population(self):
        """Initialize population within bounds"""
        population = []
        for _ in range(self.n_streams + self.n_seas + self.n_rivers):
            individual = self.lb + np.random.random(len(self.lb)) * (self.ub - self.lb)
            population.append(individual)
        return np.array(population)


class ModelManager:
    """Manages CatBoost model training and evaluation with MOWCA optimization"""
    
    def __init__(self):
        self.model: Optional[CatBoostRegressor] = None
        self.history: Optional = None
        self.feature_names: List[str] = []
        self.feature_stats: Dict = {}
        self.best_params: Dict = {}
        self.feature_importance: Optional[np.ndarray] = None
        self.is_trained = False
        
        # Default parameter ranges for MOWCA
        self.param_ranges = {
            'learning_rate': (0.01, 0.3),
            'depth': (3, 10),
            'l2_leaf_reg': (1, 10),
            'bagging_temperature': (0.1, 1.0),
            'random_strength': (0.1, 5.0),
            'border_count': (32, 255),
            'rsm': (0.5, 1.0)
        }
        
    def objective_function(self, params):
        """Objective function for MOWCA optimization"""
        try:
            # Unpack parameters
            learning_rate = params[0]
            depth = int(params[1])
            l2_leaf_reg = params[2]
            bagging_temperature = params[3]
            random_strength = params[4]
            border_count = int(params[5])
            rsm = params[6]
            
            # Create model with these parameters
            model = CatBoostRegressor(
                iterations=500,
                learning_rate=learning_rate,
                depth=depth,
                l2_leaf_reg=l2_leaf_reg,
                bagging_temperature=bagging_temperature,
                random_strength=random_strength,
                border_count=border_count,
                rsm=rsm,
                verbose=False,
                random_seed=42,
                loss_function='RMSE'
            )
            
            # Train and evaluate with cross-validation
            from sklearn.model_selection import cross_val_score
            
            # Use combined dataset for CV
            X_combined = np.vstack([self.X_train, self.X_val])
            y_combined = np.hstack([self.y_train, self.y_val])
            
            # Negative RMSE for minimization (MOWCA minimizes)
            scores = cross_val_score(model, X_combined, y_combined, 
                                    cv=3, scoring='neg_root_mean_squared_error')
            mean_score = -np.mean(scores)  # Convert to positive RMSE
            
            return mean_score
            
        except Exception as e:
            logger.warning(f"Objective evaluation failed: {e}")
            return float('inf')
    
    def train_with_mowca(self, X_train: np.ndarray, y_train: np.ndarray, 
                         X_val: np.ndarray, y_val: np.ndarray) -> Dict:
        """Train the CatBoost model using MOWCA for hyperparameter tuning"""
        
        try:
            # Store data for objective function
            self.X_train = X_train
            self.y_train = y_train
            self.X_val = X_val
            self.y_val = y_val
            
            # Define parameter bounds
            lb = [self.param_ranges[p][0] for p in ['learning_rate', 'depth', 'l2_leaf_reg',
                                                     'bagging_temperature', 'random_strength',
                                                     'border_count', 'rsm']]
            ub = [self.param_ranges[p][1] for p in ['learning_rate', 'depth', 'l2_leaf_reg',
                                                     'bagging_temperature', 'random_strength',
                                                     'border_count', 'rsm']]
            
            # Run MOWCA optimization
            logger.info("Starting MOWCA hyperparameter optimization...")
            mowca = MOWCA(
                objective_func=self.objective_function,
                n_streams=20,
                n_seas=4,
                n_rivers=4,
                max_iter=30,
                lb=lb,
                ub=ub
            )
            
            best_params_array, best_fitness, convergence = mowca.optimize()
            
            # Extract best parameters
            self.best_params = {
                'learning_rate': best_params_array[0],
                'depth': int(best_params_array[1]),
                'l2_leaf_reg': best_params_array[2],
                'bagging_temperature': best_params_array[3],
                'random_strength': best_params_array[4],
                'border_count': int(best_params_array[5]),
                'rsm': best_params_array[6]
            }
            
            logger.info(f"Best MOWCA parameters: {self.best_params}")
            
            # Train final model with best parameters
            self.model = CatBoostRegressor(
                iterations=1000,
                learning_rate=self.best_params['learning_rate'],
                depth=self.best_params['depth'],
                l2_leaf_reg=self.best_params['l2_leaf_reg'],
                bagging_temperature=self.best_params['bagging_temperature'],
                random_strength=self.best_params['random_strength'],
                border_count=self.best_params['border_count'],
                rsm=self.best_params['rsm'],
                verbose=False,
                random_seed=42,
                loss_function='RMSE',
                eval_metric='RMSE'
            )
            
            logger.info("Training final model with optimized parameters...")
            
            self.model.fit(
                X_train, y_train,
                eval_set=(X_val, y_val),
                verbose=False,
                plot=False
            )
            
            evals_result = self.model.get_evals_result()
            self.history = {
                'loss': evals_result['learn']['RMSE'],
                'val_loss': evals_result['validation']['RMSE']
            }
            
            self.feature_importance = self.model.get_feature_importance()
            self.is_trained = True
            logger.info(f"Model training completed. Best iteration: {self.model.get_best_iteration()}")
            
            return self.history
            
        except Exception as e:
            logger.error(f"Model training error: {e}")
            logger.error(traceback.format_exc())
            raise
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions"""
        if not self.is_trained or self.model is None:
            raise ValueError("Model not trained yet")
        return self.model.predict(X).flatten()


class ConcreteStrengthPredictor:
    """Main application class for Concrete Compressive Strength Prediction"""
    
    def __init__(self, data_path: str, theme: Theme = Theme.PROFESSIONAL, fullscreen: bool = True):
        self.data_path = data_path
        self.theme = theme
        self.fullscreen = fullscreen
        self.colors = ColorPalette(theme)
        
        # Data and model
        self.df = None
        self.X_raw = None
        self.y = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.X_scaled = None
        self.scaler = None
        self.feature_names = []
        self.feature_stats = {}
        self.target_name = "Cylinder compressive strength (MPa)"
        
        self.model_manager = ModelManager()
        
        # Metrics
        self.y_train_pred = None
        self.y_test_pred = None
        self.metrics = {}
        
        # History
        self.prediction_history = []
        self.current_prediction = None
        
        # GUI elements
        self.root = None
        self.entries = {}
        self.result_var = None
        self.rating_var = None
        self.status_label = None
        self.init_success = False
        
        # PDP elements
        self.pdp_fig = None
        self.pdp_canvas = None
        self.pdp_feature_var = None
        
        self._initialize()
    
    def _initialize(self):
        """Initialize application components"""
        try:
            if not self._load_and_preprocess_data():
                logger.error("Failed to load data")
                return
            
            if not self._train_model():
                logger.error("Failed to train model")
                return
            
            self.init_success = True
            self._create_gui()
            self._load_history()
        except Exception as e:
            logger.error(f"Initialization error: {e}")
            logger.error(traceback.format_exc())
    
    def _load_and_preprocess_data(self) -> bool:
        """Load and preprocess data"""
        try:
            logger.info(f"Loading data from: {self.data_path}")
            self.df = pd.read_csv(self.data_path)
            self.df.columns = self.df.columns.str.strip()
            self.df = self.df.dropna()
            
            # Find target column
            target_keywords = ['strength', 'compressive', 'cylinder', 'mpa']
            found_target = False
            
            for col in self.df.columns:
                if any(keyword in col.lower() for keyword in target_keywords):
                    self.target_name = col
                    found_target = True
                    break
            
            if not found_target:
                # Assume last column is target
                self.target_name = self.df.columns[-1]
            
            logger.info(f"Target column: '{self.target_name}'")
            
            self.X_raw = self.df.drop(columns=[self.target_name])
            self.y = self.df[self.target_name].values
            
            # Handle any NaN values
            self.y = np.nan_to_num(self.y, nan=np.median(self.y[~np.isnan(self.y)]))
            
            self.feature_names = self.X_raw.columns.tolist()
            logger.info(f"Features ({len(self.feature_names)}): {self.feature_names[:10]}...")
            
            # Calculate feature statistics
            for feature in self.feature_names:
                self.feature_stats[feature] = {
                    'min': float(self.X_raw[feature].min()),
                    'max': float(self.X_raw[feature].max()),
                    'mean': float(self.X_raw[feature].mean()),
                    'std': float(self.X_raw[feature].std()),
                    'median': float(self.X_raw[feature].median()),
                    'q1': float(self.X_raw[feature].quantile(0.25)),
                    'q3': float(self.X_raw[feature].quantile(0.75))
                }
            
            # Scale features
            self.scaler = StandardScaler()
            self.X_scaled = self.scaler.fit_transform(self.X_raw)
            
            # Split data (80-20)
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X_scaled, self.y, test_size=0.2, random_state=42
            )
            
            # Further split training for validation
            self.X_train, self.X_val, self.y_train, self.y_val = train_test_split(
                self.X_train, self.y_train, test_size=0.2, random_state=42
            )
            
            logger.info(f"Training samples: {len(self.X_train)}, Validation: {len(self.X_val)}, Testing: {len(self.X_test)}")
            logger.info(f"Target range: {self.y.min():.2f} - {self.y.max():.2f} MPa")
            return True
            
        except Exception as e:
            logger.error(f"Data loading failed: {e}")
            logger.error(traceback.format_exc())
            return False
    
    def _train_model(self) -> bool:
        """Train the model using MOWCA"""
        try:
            self.model_manager.feature_names = self.feature_names
            self.model_manager.feature_stats = self.feature_stats
            
            logger.info("Starting model training with MOWCA hyperparameter optimization...")
            history = self.model_manager.train_with_mowca(
                self.X_train, self.y_train,
                self.X_val, self.y_val
            )
            
            self._evaluate_model()
            logger.info("Model training completed successfully")
            return True
            
        except Exception as e:
            logger.error(f"Model training failed: {e}")
            logger.error(traceback.format_exc())
            return False
    
    def _evaluate_model(self):
        """Evaluate model performance"""
        try:
            self.y_train_pred = self.model_manager.predict(self.X_train)
            self.y_test_pred = self.model_manager.predict(self.X_test)
            
            # Calculate metrics
            self.metrics = {
                'R²': {
                    'Train': r2_score(self.y_train, self.y_train_pred),
                    'Test': r2_score(self.y_test, self.y_test_pred)
                },
                'RMSE': {
                    'Train': np.sqrt(mean_squared_error(self.y_train, self.y_train_pred)),
                    'Test': np.sqrt(mean_squared_error(self.y_test, self.y_test_pred))
                },
                'MAE': {
                    'Train': mean_absolute_error(self.y_train, self.y_train_pred),
                    'Test': mean_absolute_error(self.y_test, self.y_test_pred)
                },
                'MedAE': {
                    'Train': np.median(np.abs(self.y_train - self.y_train_pred)),
                    'Test': np.median(np.abs(self.y_test - self.y_test_pred))
                },
                'Max Error': {
                    'Train': max_error(self.y_train, self.y_train_pred),
                    'Test': max_error(self.y_test, self.y_test_pred)
                },
                'MAPE (%)': {
                    'Train': np.mean(np.abs((self.y_train - self.y_train_pred) / self.y_train)) * 100,
                    'Test': np.mean(np.abs((self.y_test - self.y_test_pred) / self.y_test)) * 100
                }
            }
            
            # Additional metrics
            residuals_train = self.y_train - self.y_train_pred
            residuals_test = self.y_test - self.y_test_pred
            
            # RAE (Relative Absolute Error)
            self.metrics['RAE'] = {
                'Train': np.sum(np.abs(residuals_train)) / np.sum(np.abs(self.y_train - np.mean(self.y_train))),
                'Test': np.sum(np.abs(residuals_test)) / np.sum(np.abs(self.y_test - np.mean(self.y_test)))
            }
            
            # RRSE (Root Relative Squared Error)
            self.metrics['RRSE'] = {
                'Train': np.sqrt(np.sum(residuals_train**2) / np.sum((self.y_train - np.mean(self.y_train))**2)),
                'Test': np.sqrt(np.sum(residuals_test**2) / np.sum((self.y_test - np.mean(self.y_test))**2))
            }
            
            # Willmott's Index (WI)
            self.metrics['WI'] = {
                'Train': 1 - (np.sum(residuals_train**2) / np.sum((np.abs(self.y_train_pred - np.mean(self.y_train)) + np.abs(self.y_train - np.mean(self.y_train)))**2)),
                'Test': 1 - (np.sum(residuals_test**2) / np.sum((np.abs(self.y_test_pred - np.mean(self.y_test)) + np.abs(self.y_test - np.mean(self.y_test)))**2))
            }
            
            # NSE (Nash-Sutcliffe Efficiency)
            self.metrics['NSE'] = {
                'Train': 1 - (np.sum(residuals_train**2) / np.sum((self.y_train - np.mean(self.y_train))**2)),
                'Test': 1 - (np.sum(residuals_test**2) / np.sum((self.y_test - np.mean(self.y_test))**2))
            }
            
            # Explained Variance
            self.metrics['Explained Var'] = {
                'Train': 1 - np.var(residuals_train) / np.var(self.y_train),
                'Test': 1 - np.var(residuals_test) / np.var(self.y_test)
            }
            
            logger.info(f"Model Performance:")
            logger.info(f"  R² - Train: {self.metrics['R²']['Train']:.4f}, Test: {self.metrics['R²']['Test']:.4f}")
            logger.info(f"  RMSE - Train: {self.metrics['RMSE']['Train']:.4f}, Test: {self.metrics['RMSE']['Test']:.4f}")
            logger.info(f"  MAE - Train: {self.metrics['MAE']['Train']:.4f}, Test: {self.metrics['MAE']['Test']:.4f}")
            logger.info(f"  MAPE - Train: {self.metrics['MAPE (%)']['Train']:.2f}%, Test: {self.metrics['MAPE (%)']['Test']:.2f}%")
            
        except Exception as e:
            logger.error(f"Model evaluation failed: {e}")
    
    def _create_gui(self):
        """Create the main GUI"""
        self.root = tk.Tk()
        self.root.title("🏗️ Concrete Compressive Strength Predictor - CatBoost + MOWCA AI")
        self.root.configure(bg=self.colors.bg_main)
        
        if self.fullscreen:
            self.root.attributes('-fullscreen', True)
        
        self.root.grid_rowconfigure(1, weight=1)
        self.root.grid_columnconfigure(0, weight=1)
        
        self.root.bind("<Escape>", lambda e: self._toggle_fullscreen())
        self.root.bind("<F11>", lambda e: self._toggle_fullscreen())
        
        self._create_header()
        self._create_main_content()
        self._create_status_bar()
        
        self.root.update_idletasks()
        logger.info("GUI created successfully")
    
    def _create_header(self):
        """Create modern header"""
        header = tk.Frame(self.root, height=80, bg=self.colors.primary)
        header.grid(row=0, column=0, sticky='ew')
        header.grid_propagate(False)
        header.grid_columnconfigure(0, weight=1)
        
        title = tk.Label(
            header,
            text="🏗️ Concrete Compressive Strength Predictor Pro",
            font=("Segoe UI", 20, "bold"),
            bg=self.colors.primary,
            fg='white'
        )
        title.grid(row=0, column=0, sticky='w', padx=30, pady=20)
        
        subtitle = tk.Label(
            header,
            text="Powered by CatBoost + MOWCA AI | Sustainable Concrete Design | Cylinder Strength Prediction",
            font=("Segoe UI", 10),
            bg=self.colors.primary,
            fg='white'
        )
        subtitle.grid(row=1, column=0, sticky='w', padx=30)
        
        control_frame = tk.Frame(header, bg=self.colors.primary)
        control_frame.grid(row=0, column=1, rowspan=2, sticky='e', padx=20)
        
        theme_var = tk.StringVar(value=self.theme.value)
        theme_combo = ttk.Combobox(
            control_frame, textvariable=theme_var,
            values=[t.value for t in Theme],
            state='readonly', width=12
        )
        theme_combo.pack(side='left', padx=5)
        theme_combo.bind('<<ComboboxSelected>>', lambda e: self._change_theme(theme_var.get()))
        
        fs_btn = tk.Button(
            control_frame, text="⛶", font=("Segoe UI", 14, "bold"),
            bg='white', fg=self.colors.primary, relief='flat',
            width=3, cursor='hand2', command=self._toggle_fullscreen
        )
        fs_btn.pack(side='left', padx=5)
    
    def _create_main_content(self):
        """Create main content with notebook"""
        style = ttk.Style()
        style.configure('Custom.TNotebook', background=self.colors.bg_main)
        style.configure('Custom.TNotebook.Tab', padding=[15, 8], font=('Segoe UI', 10))
        
        self.notebook = ttk.Notebook(self.root, style='Custom.TNotebook')
        self.notebook.grid(row=1, column=0, sticky='nsew', padx=15, pady=15)
        
        self._create_prediction_tab()
        self._create_performance_tab()
        self._create_pdp_tab()
        self._create_analysis_tab()
        self._create_history_tab()
    
    def _create_prediction_tab(self):
        """Create prediction tab"""
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="🎯 Predictor")
        
        paned = ttk.PanedWindow(tab, orient='horizontal')
        paned.pack(fill='both', expand=True)
        
        left_panel = tk.Frame(paned, bg=self.colors.card)
        paned.add(left_panel, weight=1)
        
        right_panel = tk.Frame(paned, bg=self.colors.card)
        paned.add(right_panel, weight=1)
        
        self._create_input_panel(left_panel)
        self._create_results_panel(right_panel)
    
    def _create_input_panel(self, parent):
        """Create input panel with proper scrolling"""
        header = tk.Frame(parent, bg=self.colors.primary, height=45)
        header.pack(fill='x')
        header.pack_propagate(False)
        
        tk.Label(
            header, text="Mix Design Parameters",
            font=("Segoe UI", 12, "bold"),
            bg=self.colors.primary, fg='white'
        ).pack(pady=10)
        
        # Create scrollable container
        scroll_container = tk.Frame(parent, bg=self.colors.card)
        scroll_container.pack(fill='both', expand=True)
        
        # Create canvas with scrollbar
        canvas = tk.Canvas(scroll_container, highlightthickness=0, bg=self.colors.card)
        scrollbar = ttk.Scrollbar(scroll_container, orient="vertical", command=canvas.yview)
        
        # Create scrollable frame
        scrollable_frame = tk.Frame(canvas, bg=self.colors.card)
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        # Pack scroll components
        scrollbar.pack(side="right", fill="y")
        canvas.pack(side="left", fill="both", expand=True)
        
        # Bind mousewheel events for smooth scrolling
        def _on_mousewheel(event):
            canvas.yview_scroll(int(-1*(event.delta/120)), "units")
        
        def _bind_mousewheel(event):
            canvas.bind_all("<MouseWheel>", _on_mousewheel)
        
        def _unbind_mousewheel(event):
            canvas.unbind_all("<MouseWheel>")
        
        canvas.bind("<Enter>", _bind_mousewheel)
        canvas.bind("<Leave>", _unbind_mousewheel)
        
        # Create preset buttons
        self._create_preset_buttons(scrollable_frame)
        
        # Create input frame
        input_frame = tk.LabelFrame(
            scrollable_frame, text="Material Composition",
            font=("Segoe UI", 10, "bold"),
            bg=self.colors.card, fg=self.colors.text_primary,
            relief='ridge', bd=1
        )
        input_frame.pack(fill='both', expand=True, padx=15, pady=10)
        
        # Configure grid
        for col in range(2):
            input_frame.grid_columnconfigure(col, weight=1)
        
        # Create input fields
        for i, feature in enumerate(self.feature_names):
            row = i // 2
            col = i % 2
            
            frame = tk.Frame(input_frame, bg=self.colors.card)
            frame.grid(row=row, column=col, sticky='ew', padx=15, pady=8)
            
            stats = self.feature_stats[feature]
            icon = self._get_feature_icon(feature)
            
            # Format feature name for display
            display_name = feature.replace('_', ' ').title()
            if len(display_name) > 25:
                display_name = display_name[:22] + "..."
            
            label = tk.Label(
                frame, text=f"{icon} {display_name}:",
                font=("Segoe UI", 9, "bold"),
                bg=self.colors.card, fg=self.colors.text_primary,
                width=22, anchor='w'
            )
            label.pack(side='left')
            
            entry = tk.Entry(
                frame, width=10, font=("Segoe UI", 9),
                relief='solid', bd=1, bg='white', fg=self.colors.primary
            )
            entry.insert(0, f"{stats['mean']:.1f}")
            entry.pack(side='left', padx=5)
            
            # Unit label
            if 'age' in feature.lower() or 'day' in feature.lower():
                unit = "days"
            elif 'water' in feature.lower():
                unit = "kg/m³"
            else:
                unit = "kg/m³"
            
            tk.Label(
                frame, text=unit, font=("Segoe UI", 8),
                bg=self.colors.card, fg=self.colors.text_secondary,
                width=5, anchor='w'
            ).pack(side='left')
            
            # Range indicator (min only to save space)
            range_text = f"[{stats['min']:.0f}]"
            tk.Label(
                frame, text=range_text,
                font=("Segoe UI", 7), fg=self.colors.text_secondary,
                bg=self.colors.card
            ).pack(side='left', padx=3)
            
            self.entries[feature] = entry
        
        # Add a hint about scrolling if there are many features
        if len(self.feature_names) > 8:
            hint_label = tk.Label(
                scrollable_frame,
                text="⬇️ Scroll down for more parameters ⬇️",
                font=("Segoe UI", 9, "italic"),
                bg=self.colors.card, fg=self.colors.text_secondary
            )
            hint_label.pack(pady=5)
        
        # Predict button
        predict_btn = tk.Button(
            scrollable_frame,
            text="🔮 PREDICT STRENGTH",
            font=("Segoe UI", 13, "bold"),
            bg=self.colors.success, fg='white',
            relief='flat', cursor='hand2',
            padx=20, pady=10,
            command=self._threaded_predict
        )
        predict_btn.pack(pady=15, padx=20, fill='x')
        
        # Add spacer
        spacer = tk.Frame(scrollable_frame, height=20, bg=self.colors.card)
        spacer.pack()
    
    def _create_preset_buttons(self, parent):
        """Create preset buttons"""
        preset_frame = tk.LabelFrame(
            parent, text="Quick Mix Presets",
            font=("Segoe UI", 10, "bold"),
            bg=self.colors.card, fg=self.colors.text_primary,
            relief='ridge', bd=1
        )
        preset_frame.pack(fill='x', padx=15, pady=10)
        
        button_frame = tk.Frame(preset_frame, bg=self.colors.card)
        button_frame.pack(pady=12)
        
        presets = [
            ("💪 High Strength", "high", self.colors.success),
            ("⚖️ Standard Mix", "medium", self.colors.warning),
            ("🌱 Low Carbon", "low", self.colors.danger),
            ("🎲 Random", "random", self.colors.secondary)
        ]
        
        for text, ptype, color in presets:
            btn = tk.Button(
                button_frame, text=text, font=("Segoe UI", 10, "bold"),
                bg=color, fg='white', relief='flat',
                padx=15, pady=8, cursor='hand2',
                command=lambda t=ptype: self._load_preset(t)
            )
            btn.pack(side='left', padx=8)
    
    def _create_results_panel(self, parent):
        """Create results panel"""
        # Create scrollable results panel
        scroll_container = tk.Frame(parent, bg=self.colors.card)
        scroll_container.pack(fill='both', expand=True)
        
        canvas = tk.Canvas(scroll_container, highlightthickness=0, bg=self.colors.card)
        scrollbar = ttk.Scrollbar(scroll_container, orient="vertical", command=canvas.yview)
        
        scrollable_frame = tk.Frame(canvas, bg=self.colors.card)
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        scrollbar.pack(side="right", fill="y")
        canvas.pack(side="left", fill="both", expand=True)
        
        # Bind mousewheel
        def _on_mousewheel(event):
            canvas.yview_scroll(int(-1*(event.delta/120)), "units")
        
        canvas.bind("<Enter>", lambda e: canvas.bind_all("<MouseWheel>", _on_mousewheel))
        canvas.bind("<Leave>", lambda e: canvas.unbind_all("<MouseWheel>"))
        
        result_card = tk.Frame(scrollable_frame, bg=self.colors.card, relief='ridge', bd=1)
        result_card.pack(fill='x', padx=20, pady=20)
        
        tk.Label(
            result_card, text="Predicted Compressive Strength",
            font=("Segoe UI", 14, "bold"),
            bg=self.colors.card, fg=self.colors.text_primary
        ).pack(pady=(20, 5))
        
        self.result_var = tk.StringVar(value="---")
        result_label = tk.Label(
            result_card, textvariable=self.result_var,
            font=("Segoe UI", 64, "bold"),
            fg=self.colors.success, bg=self.colors.card
        )
        result_label.pack(pady=10)
        
        tk.Label(
            result_card, text="Megapascals (MPa)",
            font=("Segoe UI", 10),
            fg=self.colors.text_secondary, bg=self.colors.card
        ).pack(pady=(0, 20))
        
        self.rating_var = tk.StringVar(value="")
        rating_label = tk.Label(
            scrollable_frame, textvariable=self.rating_var,
            font=("Segoe UI", 12, "bold"),
            bg=self.colors.card
        )
        rating_label.pack(pady=10)
        
        btn_frame = tk.Frame(scrollable_frame, bg=self.colors.card)
        btn_frame.pack(fill='x', pady=15, padx=20)
        
        actions = [
            ("💾 Save", self._save_result, self.colors.success),
            ("📋 Copy", self._copy_result, self.colors.primary),
            ("📊 Export", self._export_data, self.colors.secondary),
            ("🔄 Compare", self._compare_baseline, self.colors.warning)
        ]
        
        for text, cmd, color in actions:
            btn = tk.Button(
                btn_frame, text=text, font=("Segoe UI", 10, "bold"),
                bg=color, fg='white', relief='flat',
                padx=20, pady=8, cursor='hand2', command=cmd
            )
            btn.pack(side='left', padx=5, expand=True, fill='x')
        
        # Show best MOWCA parameters
        if self.model_manager.best_params:
            params_frame = tk.LabelFrame(
                scrollable_frame, text="🔧 MOWCA Optimized Parameters",
                font=("Segoe UI", 10, "bold"),
                bg=self.colors.card, fg=self.colors.text_primary,
                relief='ridge', bd=1
            )
            params_frame.pack(fill='x', padx=20, pady=10)
            
            params_text = ""
            for param, value in self.model_manager.best_params.items():
                if isinstance(value, float):
                    params_text += f"{param}: {value:.4f}  |  "
                else:
                    params_text += f"{param}: {value}  |  "
            
            tk.Label(
                params_frame, text=params_text,
                font=("Segoe UI", 9),
                bg=self.colors.card, fg=self.colors.text_secondary,
                wraplength=400
            ).pack(pady=10, padx=10)
    
    def _create_performance_tab(self):
        """Create performance tab with metrics display"""
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="📈 Performance")
        
        main_frame = tk.Frame(tab, bg=self.colors.bg_main)
        main_frame.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Title header
        header_frame = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=1)
        header_frame.pack(fill='x', pady=(0, 10))
        
        tk.Label(
            header_frame,
            text="🎯 Model Performance Dashboard",
            font=("Segoe UI", 14, "bold"),
            bg=self.colors.card,
            fg=self.colors.text_primary
        ).pack(pady=10)
        
        # Create metrics display in a table format
        if self.metrics:
            metrics_frame = tk.Frame(header_frame, bg=self.colors.card)
            metrics_frame.pack(fill='x', pady=10, padx=20)
            
            # Header row
            headers = ["Metric", "Train", "Test"]
            for i, header in enumerate(headers):
                lbl = tk.Label(
                    metrics_frame, text=header,
                    font=("Segoe UI", 11, "bold"),
                    bg=self.colors.primary, fg='white',
                    padx=15, pady=5
                )
                lbl.grid(row=0, column=i, sticky='ew', padx=1)
            
            # Metric rows
            row = 1
            for metric_name in ['R²', 'RMSE', 'MAE', 'MedAE', 'Max Error', 'RAE', 'RRSE', 'WI', 'NSE', 'Explained Var', 'MAPE (%)']:
                if metric_name in self.metrics:
                    metric_lbl = tk.Label(
                        metrics_frame, text=metric_name,
                        font=("Segoe UI", 10),
                        bg=self.colors.card, fg=self.colors.text_primary,
                        padx=15, pady=3
                    )
                    metric_lbl.grid(row=row, column=0, sticky='w', padx=1)
                    
                    train_val = self.metrics[metric_name]['Train']
                    test_val = self.metrics[metric_name]['Test']
                    
                    if isinstance(train_val, float):
                        train_str = f"{train_val:.4f}"
                        test_str = f"{test_val:.4f}"
                    else:
                        train_str = str(train_val)
                        test_str = str(test_val)
                    
                    train_lbl = tk.Label(
                        metrics_frame, text=train_str,
                        font=("Segoe UI", 10),
                        bg=self.colors.card, fg=self.colors.text_primary,
                        padx=15, pady=3
                    )
                    train_lbl.grid(row=row, column=1, sticky='e', padx=1)
                    
                    test_lbl = tk.Label(
                        metrics_frame, text=test_str,
                        font=("Segoe UI", 10),
                        bg=self.colors.card, fg=self.colors.text_primary,
                        padx=15, pady=3
                    )
                    test_lbl.grid(row=row, column=2, sticky='e', padx=1)
                    
                    row += 1
        
        # Create 2x2 grid for plots
        for i in range(2):
            main_frame.grid_rowconfigure(i, weight=1)
            main_frame.grid_columnconfigure(i, weight=1)
        
        # Frame 1: Actual vs Predicted
        frame1 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame1.grid(row=0, column=0, sticky='nsew', padx=5, pady=5)
        
        header1 = tk.Frame(frame1, bg=self.colors.primary, height=35)
        header1.pack(fill='x')
        header1.pack_propagate(False)
        tk.Label(header1, text="🎯 Actual vs Predicted Strength", font=("Segoe UI", 10, "bold"),
                bg=self.colors.primary, fg='white').pack(pady=5)
        
        self._plot_actual_vs_predicted(frame1)
        
        # Frame 2: Residuals
        frame2 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame2.grid(row=0, column=1, sticky='nsew', padx=5, pady=5)
        
        header2 = tk.Frame(frame2, bg=self.colors.secondary, height=35)
        header2.pack(fill='x')
        header2.pack_propagate(False)
        tk.Label(header2, text="📊 Residual Analysis", font=("Segoe UI", 10, "bold"),
                bg=self.colors.secondary, fg='white').pack(pady=5)
        
        self._plot_residuals(frame2)
        
        # Frame 3: Training History
        frame3 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame3.grid(row=1, column=0, sticky='nsew', padx=5, pady=5)
        
        header3 = tk.Frame(frame3, bg=self.colors.success, height=35)
        header3.pack(fill='x')
        header3.pack_propagate(False)
        tk.Label(header3, text="📈 Training Progress", font=("Segoe UI", 10, "bold"),
                bg=self.colors.success, fg='white').pack(pady=5)
        
        self._plot_training_history(frame3)
        
        # Frame 4: Error Distribution
        frame4 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame4.grid(row=1, column=1, sticky='nsew', padx=5, pady=5)
        
        header4 = tk.Frame(frame4, bg=self.colors.warning, height=35)
        header4.pack(fill='x')
        header4.pack_propagate(False)
        tk.Label(header4, text="⚠️ Error Distribution", font=("Segoe UI", 10, "bold"),
                bg=self.colors.warning, fg='white').pack(pady=5)
        
        self._plot_error_distribution(frame4)
    
    def _plot_actual_vs_predicted(self, parent):
        """Plot actual vs predicted"""
        if not self.model_manager.is_trained:
            return
            
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        ax.scatter(self.y_test, self.y_test_pred, alpha=0.6, 
                  c=self.colors.primary, edgecolors='black', linewidth=0.5, s=50)
        
        min_val = min(self.y_test.min(), self.y_test_pred.min())
        max_val = max(self.y_test.max(), self.y_test_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
        
        # Add R² and RMSE text box
        textstr = f'R² = {self.metrics["R²"]["Test"]:.4f}\nRMSE = {self.metrics["RMSE"]["Test"]:.4f} MPa'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=9,
                verticalalignment='top', bbox=props)
        
        ax.set_xlabel('Actual Strength (MPa)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Predicted Strength (MPa)', fontsize=10, fontweight='bold')
        ax.set_title('Model Predictions vs Actual Values', fontsize=11, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _plot_residuals(self, parent):
        """Plot residuals"""
        if not self.model_manager.is_trained:
            return
            
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        residuals = self.y_test - self.y_test_pred
        ax.scatter(self.y_test_pred, residuals, alpha=0.6, 
                  c=self.colors.secondary, edgecolors='black', linewidth=0.5, s=50)
        ax.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
        
        mean_res = np.mean(residuals)
        std_res = np.std(residuals)
        textstr = f'Mean Residual: {mean_res:.4f} MPa\nStd Residual: {std_res:.4f} MPa'
        props = dict(boxstyle='round', facecolor='lightblue', alpha=0.8)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=9,
                verticalalignment='top', bbox=props)
        
        ax.set_xlabel('Predicted Strength (MPa)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Residuals (MPa)', fontsize=10, fontweight='bold')
        ax.set_title('Residual Plot - Error Analysis', fontsize=11, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _plot_training_history(self, parent):
        """Plot training history"""
        if not self.model_manager.history:
            return
            
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        if self.model_manager.history:
            epochs = range(1, len(self.model_manager.history['loss']) + 1)
            ax.plot(epochs, self.model_manager.history['loss'], 
                   label='Training', linewidth=2, color=self.colors.primary)
            ax.plot(epochs, self.model_manager.history['val_loss'], 
                   label='Validation', linewidth=2, color=self.colors.secondary)
            
            best_iter = self.model_manager.model.get_best_iteration()
            ax.axvline(x=best_iter, color='red', linestyle='--', alpha=0.7, 
                      label=f'Best Iteration: {best_iter}')
            
            final_train = self.model_manager.history['loss'][-1]
            final_val = self.model_manager.history['val_loss'][-1]
            textstr = f'Final Train RMSE: {final_train:.4f}\nFinal Val RMSE: {final_val:.4f}'
            props = dict(boxstyle='round', facecolor='lightgreen', alpha=0.8)
            ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=8,
                    verticalalignment='top', horizontalalignment='right', bbox=props)
            
            ax.set_xlabel('Iteration', fontsize=10, fontweight='bold')
            ax.set_ylabel('RMSE (MPa)', fontsize=10, fontweight='bold')
            ax.set_title('Model Training Progress', fontsize=11, fontweight='bold')
            ax.legend(loc='upper right')
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _plot_error_distribution(self, parent):
        """Plot error distribution"""
        if not self.model_manager.is_trained:
            return
            
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        residuals = self.y_test - self.y_test_pred
        
        ax.hist(residuals, bins=30, edgecolor='black', 
               alpha=0.7, color=self.colors.primary,
               density=True, linewidth=1.5)
        
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2.5, label='Zero Error')
        
        mean_err = np.mean(residuals)
        std_err = np.std(residuals)
        ax.axvline(mean_err, color='blue', linestyle='-', linewidth=2, 
                  alpha=0.7, label=f'Mean: {mean_err:.4f}')
        ax.axvline(mean_err + std_err, color='green', linestyle=':', 
                  linewidth=1.5, alpha=0.7, label=f'±1σ: {std_err:.4f}')
        ax.axvline(mean_err - std_err, color='green', linestyle=':', 
                  linewidth=1.5, alpha=0.7)
        
        stats_text = f'Error Statistics:\nMean: {mean_err:.4f}\nStd: {std_err:.4f}\nMAE: {self.metrics["MAE"]["Test"]:.4f}'
        ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, 
               fontsize=8, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        ax.set_xlabel('Prediction Error (MPa)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Density', fontsize=10, fontweight='bold')
        ax.set_title('Error Distribution Analysis', fontsize=11, fontweight='bold')
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _create_pdp_tab(self):
        """Create Partial Dependence Plot tab"""
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="📈 PDP Analysis")
        
        main_frame = tk.Frame(tab, bg=self.colors.bg_main)
        main_frame.pack(fill='both', expand=True, padx=10, pady=10)
        
        control_panel = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=1)
        control_panel.pack(fill='x', pady=(0, 10))
        
        tk.Label(
            control_panel, text="Partial Dependence Analysis",
            font=("Segoe UI", 12, "bold"),
            bg=self.colors.card, fg=self.colors.text_primary
        ).pack(pady=10)
        
        select_frame = tk.Frame(control_panel, bg=self.colors.card)
        select_frame.pack(pady=10)
        
        tk.Label(
            select_frame, text="Select Feature:", font=("Segoe UI", 10, "bold"),
            bg=self.colors.card, fg=self.colors.text_primary
        ).pack(side='left', padx=10)
        
        self.pdp_feature_var = tk.StringVar()
        pdp_combo = ttk.Combobox(
            select_frame, textvariable=self.pdp_feature_var,
            values=self.feature_names, state='readonly', width=25
        )
        pdp_combo.pack(side='left', padx=10)
        if self.feature_names:
            pdp_combo.set(self.feature_names[0])
        pdp_combo.bind('<<ComboboxSelected>>', lambda e: self._update_pdp_plot())
        
        info_label = tk.Label(
            control_panel,
            text="💡 PDP shows how the predicted strength changes as you vary the selected feature",
            font=("Segoe UI", 9, "italic"),
            bg=self.colors.card, fg=self.colors.text_secondary
        )
        info_label.pack(pady=10)
        
        self.pdp_plot_frame = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=1)
        self.pdp_plot_frame.pack(fill='both', expand=True)
        
        self._update_pdp_plot()
    
    def _update_pdp_plot(self):
        """Update Partial Dependence Plot"""
        if not self.model_manager.is_trained:
            return
            
        feature = self.pdp_feature_var.get()
        if not feature:
            return
        
        for widget in self.pdp_plot_frame.winfo_children():
            widget.destroy()
        
        try:
            fig, ax = plt.subplots(figsize=(8, 6), facecolor='white', dpi=100)
            self._plot_single_pdp(feature, ax)
            
            canvas = FigureCanvasTkAgg(fig, self.pdp_plot_frame)
            canvas.draw()
            canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
            
        except Exception as e:
            logger.error(f"PDP plot error: {e}")
    
    def _plot_single_pdp(self, feature: str, ax):
        """Plot single feature PDP"""
        stats = self.feature_stats[feature]
        feature_range = np.linspace(stats['min'], stats['max'], 50)
        predictions = []
        
        feature_idx = self.feature_names.index(feature)
        base_input = np.array([[self.feature_stats[f]['mean'] for f in self.feature_names]])
        
        for value in feature_range:
            input_data = base_input.copy()
            input_data[0][feature_idx] = value
            input_scaled = self.scaler.transform(input_data)
            pred = self.model_manager.predict(input_scaled)[0]
            predictions.append(pred)
        
        ax.plot(feature_range, predictions, linewidth=3, color=self.colors.primary, label='Predicted Strength')
        ax.fill_between(feature_range, 
                       np.array(predictions) - np.std(predictions),
                       np.array(predictions) + np.std(predictions),
                       alpha=0.2, color=self.colors.primary)
        
        opt_idx = np.argmax(predictions)  # Maximize strength
        opt_value = feature_range[opt_idx]
        opt_pred = predictions[opt_idx]
        
        ax.plot(opt_value, opt_pred, 'go', markersize=12, label='Optimal Point', zorder=5)
        ax.axvline(x=opt_value, color='green', linestyle='--', alpha=0.7, linewidth=2)
        
        unit = "days" if 'age' in feature.lower() else "kg/m³"
        ax.set_xlabel(f"{feature} ({unit})", fontsize=11, fontweight='bold')
        ax.set_ylabel('Compressive Strength (MPa)', fontsize=11, fontweight='bold')
        ax.set_title(f'Partial Dependence Plot: {feature}', fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=10)
        
        ax.annotate(f'Optimal: {opt_value:.1f} {unit}\nStrength: {opt_pred:.1f} MPa',
                   xy=(opt_value, opt_pred),
                   xytext=(opt_value + (stats['max']-stats['min'])*0.1, opt_pred + 5),
                   arrowprops=dict(arrowstyle='->', color='green', lw=2),
                   fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        
        ax2 = ax.twinx()
        ax2.hist(self.X_raw[feature], bins=20, alpha=0.3, color=self.colors.secondary)
        ax2.set_ylabel('Data Distribution', fontsize=9, color=self.colors.secondary)
        ax2.tick_params(axis='y', labelcolor=self.colors.secondary)
        
        plt.tight_layout()
    
    def _create_analysis_tab(self):
        """Create analysis tab"""
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="🔍 Analysis")
        
        main_frame = tk.Frame(tab, bg=self.colors.bg_main)
        main_frame.pack(fill='both', expand=True, padx=10, pady=10)
        
        title_frame = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=1)
        title_frame.pack(fill='x', pady=(0, 10))
        
        title_label = tk.Label(
            title_frame,
            text="📊 Advanced Data Analysis Dashboard",
            font=("Segoe UI", 14, "bold"),
            bg=self.colors.card,
            fg=self.colors.text_primary
        )
        title_label.pack(pady=10)
        
        description_label = tk.Label(
            title_frame,
            text="Explore feature correlations, importance rankings, and strength distributions",
            font=("Segoe UI", 10),
            bg=self.colors.card,
            fg=self.colors.text_secondary
        )
        description_label.pack(pady=(0, 10))
        
        for i in range(2):
            main_frame.grid_rowconfigure(i, weight=1)
            main_frame.grid_columnconfigure(i, weight=1)
        
        # Frame 1: Correlation Heatmap
        frame1 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame1.grid(row=0, column=0, sticky='nsew', padx=5, pady=5)
        
        header1 = tk.Frame(frame1, bg=self.colors.primary, height=35)
        header1.pack(fill='x')
        header1.pack_propagate(False)
        tk.Label(header1, text="📈 Feature Correlation Matrix", font=("Segoe UI", 10, "bold"),
                bg=self.colors.primary, fg='white').pack(pady=5)
        
        self._plot_correlation_heatmap(frame1)
        
        # Frame 2: Feature Importance
        frame2 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame2.grid(row=0, column=1, sticky='nsew', padx=5, pady=5)
        
        header2 = tk.Frame(frame2, bg=self.colors.secondary, height=35)
        header2.pack(fill='x')
        header2.pack_propagate(False)
        tk.Label(header2, text="⭐ CatBoost Feature Importance", font=("Segoe UI", 10, "bold"),
                bg=self.colors.secondary, fg='white').pack(pady=5)
        
        self._plot_feature_importance(frame2)
        
        # Frame 3: Strength Distribution
        frame3 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame3.grid(row=1, column=0, sticky='nsew', padx=5, pady=5)
        
        header3 = tk.Frame(frame3, bg=self.colors.success, height=35)
        header3.pack(fill='x')
        header3.pack_propagate(False)
        tk.Label(header3, text="📊 Strength Distribution Analysis", font=("Segoe UI", 10, "bold"),
                bg=self.colors.success, fg='white').pack(pady=5)
        
        self._plot_strength_distribution(frame3)
        
        # Frame 4: Prediction vs Actual Distribution
        frame4 = tk.Frame(main_frame, bg=self.colors.card, relief='ridge', bd=2)
        frame4.grid(row=1, column=1, sticky='nsew', padx=5, pady=5)
        
        header4 = tk.Frame(frame4, bg=self.colors.warning, height=35)
        header4.pack(fill='x')
        header4.pack_propagate(False)
        tk.Label(header4, text="📊 Prediction Distribution", font=("Segoe UI", 10, "bold"),
                bg=self.colors.warning, fg='white').pack(pady=5)
        
        self._plot_prediction_distribution(frame4)
    
    def _plot_correlation_heatmap(self, parent):
        """Plot correlation heatmap"""
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        # Calculate correlation with target
        corr_with_target = self.X_raw.corrwith(self.df[self.target_name]).sort_values(ascending=False)
        
        # Show top 15 features
        top_features = corr_with_target.head(15).index
        X_top = self.X_raw[top_features]
        corr_matrix = X_top.corr()
        
        im = ax.imshow(corr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        
        cbar = plt.colorbar(im, ax=ax, shrink=0.8)
        cbar.set_label('Correlation Coefficient', fontsize=9, fontweight='bold')
        
        ax.set_xticks(range(len(corr_matrix.columns)))
        ax.set_yticks(range(len(corr_matrix.columns)))
        ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(corr_matrix.columns, fontsize=7)
        
        ax.set_title('Feature Correlation Matrix', fontsize=11, fontweight='bold', pad=20)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _plot_feature_importance(self, parent):
        """Plot CatBoost feature importance"""
        if self.model_manager.feature_importance is None:
            return
            
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
    
        importance = self.model_manager.feature_importance
        indices = np.argsort(importance)[::-1][:10]
        
        colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(indices)))
        y_pos = np.arange(len(indices))
        bars = ax.barh(y_pos, importance[indices], color=colors)
        
        for i, (bar, val) in enumerate(zip(bars, importance[indices])):
            width = bar.get_width()
            ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{val:.4f}', ha='left', va='center', fontsize=8, fontweight='bold')
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels([self.feature_names[i][:20] for i in indices], fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('Feature Importance Score', fontsize=10, fontweight='bold')
        ax.set_title('Top 10 Most Important Features', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _plot_strength_distribution(self, parent):
        """Plot strength distribution"""
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        ax.hist(self.y, bins=30, alpha=0.7, color=self.colors.primary,
               edgecolor='black', linewidth=1.5, density=True)
        
        mean_val = np.mean(self.y)
        median_val = np.median(self.y)
        std_val = np.std(self.y)
        
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, 
                  alpha=0.8, label=f'Mean: {mean_val:.1f} MPa')
        ax.axvline(median_val, color='green', linestyle='--', linewidth=2, 
                  alpha=0.8, label=f'Median: {median_val:.1f} MPa')
        
        stats_text = f'Statistics:\nMean: {mean_val:.1f}\nMedian: {median_val:.1f}\nStd: {std_val:.1f}'
        ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, 
               fontsize=8, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax.set_xlabel('Compressive Strength (MPa)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Density', fontsize=10, fontweight='bold')
        ax.set_title('Target Variable Distribution', fontsize=11, fontweight='bold')
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _plot_prediction_distribution(self, parent):
        """Plot prediction distribution"""
        if not self.model_manager.is_trained:
            return
            
        fig, ax = plt.subplots(figsize=(6, 5), facecolor='white', dpi=100)
        
        for widget in parent.winfo_children():
            if isinstance(widget, FigureCanvasTkAgg):
                widget.destroy()
        
        ax.hist(self.y_test_pred, bins=30, alpha=0.7, color=self.colors.primary,
               edgecolor='black', linewidth=1.5, label='Predictions', density=True)
        ax.hist(self.y_test, bins=30, alpha=0.5, color=self.colors.secondary,
               edgecolor='black', linewidth=1.5, label='Actual', density=True)
        
        pred_mean = np.mean(self.y_test_pred)
        actual_mean = np.mean(self.y_test)
        
        ax.axvline(pred_mean, color=self.colors.primary, linestyle='--', 
                  linewidth=2, alpha=0.8, label=f'Pred Mean: {pred_mean:.1f}')
        ax.axvline(actual_mean, color=self.colors.secondary, linestyle='--', 
                  linewidth=2, alpha=0.8, label=f'Actual Mean: {actual_mean:.1f}')
        
        stats_text = f'Stats:\nPred Std: {np.std(self.y_test_pred):.1f}\nActual Std: {np.std(self.y_test):.1f}'
        ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, 
               fontsize=8, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax.set_xlabel('Strength (MPa)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Density', fontsize=10, fontweight='bold')
        ax.set_title('Distribution Comparison: Predicted vs Actual', fontsize=11, fontweight='bold')
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        canvas = FigureCanvasTkAgg(fig, parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True, padx=5, pady=5)
    
    def _create_history_tab(self):
        """Create history tab"""
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="📜 History")
        
        container = tk.Frame(tab, bg=self.colors.bg_main)
        container.pack(fill='both', expand=True, padx=10, pady=10)
        
        self.history_stats = tk.Label(
            container, text="", font=("Segoe UI", 10),
            bg=self.colors.card, fg=self.colors.text_primary,
            relief='ridge', bd=1
        )
        self.history_stats.pack(fill='x', pady=(0, 10))
        
        tree_frame = tk.Frame(container, bg=self.colors.card)
        tree_frame.pack(fill='both', expand=True)
        
        columns = ('Timestamp', 'Strength (MPa)', 'Rating', 'Age (days)')
        
        vsb = ttk.Scrollbar(tree_frame, orient="vertical")
        hsb = ttk.Scrollbar(tree_frame, orient="horizontal")
        self.history_tree = ttk.Treeview(
            tree_frame, columns=columns, show='headings',
            yscrollcommand=vsb.set, xscrollcommand=hsb.set
        )
        
        vsb.config(command=self.history_tree.yview)
        hsb.config(command=self.history_tree.xview)
        
        self.history_tree.grid(row=0, column=0, sticky='nsew')
        vsb.grid(row=0, column=1, sticky='ns')
        hsb.grid(row=1, column=0, sticky='ew')
        
        tree_frame.grid_rowconfigure(0, weight=1)
        tree_frame.grid_columnconfigure(0, weight=1)
        
        for col in columns:
            self.history_tree.heading(col, text=col)
            self.history_tree.column(col, width=150)
        
        btn_frame = tk.Frame(container, bg=self.colors.bg_main)
        btn_frame.pack(pady=10)
        
        for text, cmd, color in [("🗑️ Clear", self._clear_history, self.colors.danger),
                                  ("📥 Export", self._export_history, self.colors.success)]:
            btn = tk.Button(
                btn_frame, text=text, command=cmd,
                bg=color, fg='white', relief='flat',
                padx=20, pady=5, cursor='hand2'
            )
            btn.pack(side='left', padx=5)
        
        self._update_history_display()
    
    def _create_status_bar(self):
        """Create status bar"""
        self.status_bar = tk.Frame(self.root, height=35, bg=self.colors.border)
        self.status_bar.grid(row=2, column=0, sticky='ew')
        
        self.status_label = tk.Label(
            self.status_bar, text="✅ System Ready | CatBoost + MOWCA Optimized",
            font=("Segoe UI", 9), bg=self.colors.border,
            fg=self.colors.text_secondary
        )
        self.status_label.pack(side='left', padx=10, pady=5)
        
        version_label = tk.Label(
            self.status_bar, text="🏗️ Concrete Strength Predictor v4.0 | CatBoost + MOWCA | Press F11 for Fullscreen",
            font=("Segoe UI", 9), bg=self.colors.border,
            fg=self.colors.text_secondary
        )
        version_label.pack(side='right', padx=10, pady=5)
    
    def _get_feature_icon(self, feature: str) -> str:
        """Get icon for feature"""
        icons = {
            'cement': '🏭', 'slag': '🏭', 'fly': '🌋', 'ash': '🌋',
            'water': '💧', 'superplasticizer': '✨', 'aggregate': '🪨',
            'sand': '🏖️', 'age': '📅', 'curing': '🌡️'
        }
        for key, icon in icons.items():
            if key.lower() in feature.lower():
                return icon
        return '📊'
    
    def _load_preset(self, preset_type: str):
        """Load preset values"""
        if preset_type == "random":
            for feature, entry in self.entries.items():
                stats = self.feature_stats[feature]
                value = np.random.uniform(stats['min'], stats['max'])
                entry.delete(0, tk.END)
                entry.insert(0, f"{value:.1f}")
            self._update_status("🎲 Random mix generated")
            return
        
        presets = {
            'high': {'cement': 450, 'water': 160, 'superplasticizer': 8, 'age': 28},
            'medium': {'cement': 350, 'water': 180, 'superplasticizer': 5, 'age': 28},
            'low': {'cement': 250, 'water': 200, 'superplasticizer': 2, 'age': 28}
        }
        
        if preset_type in presets:
            for feature, entry in self.entries.items():
                for p_feature, value in presets[preset_type].items():
                    if p_feature.lower() in feature.lower():
                        entry.delete(0, tk.END)
                        entry.insert(0, str(value))
                        break
            self._update_status(f"✅ Loaded {preset_type} strength preset")
    
    def _threaded_predict(self):
        """Run prediction in thread"""
        if not self.model_manager.is_trained:
            self._update_status("❌ Model not trained yet. Please wait...")
            return
            
        thread = threading.Thread(target=self._predict)
        thread.daemon = True
        thread.start()
        self._update_status("🔮 AI is analyzing your mix design...")
    
    def _predict(self):
        """Make prediction"""
        try:
            inputs = {}
            for feature, entry in self.entries.items():
                val = entry.get().strip()
                if val:
                    inputs[feature] = float(val)
                else:
                    inputs[feature] = self.feature_stats[feature]['mean']
            
            input_array = np.array([[inputs[f] for f in self.feature_names]])
            input_scaled = self.scaler.transform(input_array)
            prediction = self.model_manager.predict(input_scaled)[0]
            
            self.root.after(0, lambda: self._display_result(prediction, inputs))
            
        except Exception as e:
            error_msg = f"Prediction failed: {str(e)}"
            logger.error(error_msg)
            self.root.after(0, lambda: self._update_status(f"❌ {error_msg}"))
    
    def _display_result(self, prediction: float, inputs: Dict):
        """Display prediction result"""
        self.current_prediction = prediction
        
        self._animate_result(prediction)
        
        if prediction >= 50:
            rating = "🌟 EXCELLENT - High Strength Concrete!"
            emoji = "🏗️"
            recommendation = "Great performance! This mix meets high-strength requirements."
        elif prediction >= 30:
            rating = "👍 GOOD - Moderate Strength Concrete"
            emoji = "⚖️"
            recommendation = "Good! Consider adjusting cement content for higher strength."
        else:
            rating = "⚠️ LOW - Needs Optimization"
            emoji = "⚠️"
            recommendation = "Optimization needed. Try high-strength preset for better results."
        
        self.rating_var.set(f"{emoji} {rating}\n\n💡 {recommendation}")
        
        history_entry = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'strength': float(prediction),
            'rating': rating,
            **inputs
        }
        self.prediction_history.append(history_entry)
        self._save_history()
        self._update_history_display()
        
        self._update_status(f"✅ Prediction complete! Strength: {prediction:.2f} MPa")
    
    def _animate_result(self, target_value: float):
        """Animate result display"""
        current = 0
        steps = 20
        increment = target_value / steps
        
        def update_step(step):
            if step <= steps:
                self.result_var.set(f"{current + increment * step:.1f}")
                self.root.after(20, lambda: update_step(step + 1))
            else:
                self.result_var.set(f"{target_value:.1f}")
        
        update_step(1)
    
    def _save_result(self):
        """Save result to file"""
        if self.current_prediction is None:
            self._update_status("⚠️ Make a prediction first")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("CSV files", "*.csv")],
            initialfile=f"strength_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        )
        
        if filename:
            with open(filename, 'w') as f:
                f.write("="*70 + "\n")
                f.write("🏗️ CONCRETE STRENGTH PREDICTION REPORT\n")
                f.write("="*70 + "\n")
                f.write(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"🎯 Predicted Strength: {self.result_var.get()} MPa\n")
                f.write(f"⭐ Rating: {self.rating_var.get()}\n\n")
                f.write("📊 INPUT PARAMETERS:\n")
                f.write("-"*50 + "\n")
                for feature, entry in self.entries.items():
                    f.write(f"  {feature:30}: {entry.get():>10}\n")
                f.write("\n" + "="*70 + "\n")
            self._update_status(f"📄 Report saved")
    
    def _copy_result(self):
        """Copy result to clipboard"""
        text = f"""🏗️ Concrete Strength Prediction
{'='*40}
📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
🎯 Predicted Strength: {self.result_var.get()} MPa
⭐ Rating: {self.rating_var.get()}
{'='*40}"""
        self.root.clipboard_clear()
        self.root.clipboard_append(text)
        self._update_status("📋 Result copied to clipboard!")
    
    def _export_data(self):
        """Export prediction data"""
        if not self.prediction_history:
            self._update_status("⚠️ No data to export")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv")],
            initialfile=f"strength_predictions_{datetime.now().strftime('%Y%m%d')}.csv"
        )
        
        if filename:
            df = pd.DataFrame(self.prediction_history)
            df.to_csv(filename, index=False)
            self._update_status(f"📊 Data exported")
    
    def _compare_baseline(self):
        """Compare with baseline"""
        if self.current_prediction is None:
            self._update_status("⚠️ Make a prediction first")
            return
        
        baseline = 35  # Standard concrete strength (MPa)
        improvement = ((self.current_prediction - baseline) / baseline) * 100
        
        if improvement > 0:
            message = f"🎉 {improvement:.1f}% stronger than standard concrete!"
        else:
            message = f"⚠️ {abs(improvement):.1f}% weaker than standard concrete"
        
        self._update_status(message)
    
    def _clear_history(self):
        """Clear prediction history"""
        self.prediction_history = []
        self._save_history()
        self._update_history_display()
        self._update_status("🗑️ History cleared")
    
    def _export_history(self):
        """Export history to CSV"""
        if not self.prediction_history:
            self._update_status("⚠️ No history to export")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv")],
            initialfile=f"prediction_history_{datetime.now().strftime('%Y%m%d')}.csv"
        )
        
        if filename:
            df = pd.DataFrame(self.prediction_history)
            df.to_csv(filename, index=False)
            self._update_status(f"📊 History exported")
    
    def _update_history_display(self):
        """Update history display"""
        if hasattr(self, 'history_tree'):
            for item in self.history_tree.get_children():
                self.history_tree.delete(item)
            
            for entry in reversed(self.prediction_history[-50:]):
                # Find age column
                age_value = ""
                for key in entry.keys():
                    if 'age' in key.lower():
                        age_value = entry.get(key, '')
                        break
                
                self.history_tree.insert('', 'end', values=(
                    entry.get('timestamp', ''),
                    f"{entry.get('strength', 0):.1f}",
                    entry.get('rating', '')[:30],
                    age_value
                ))
            
            if self.prediction_history:
                strength_values = [h.get('strength', 0) for h in self.prediction_history]
                stats_text = f"📊 {len(self.prediction_history)} predictions | "
                stats_text += f"Avg: {np.mean(strength_values):.1f} | "
                stats_text += f"Min: {np.min(strength_values):.1f} | "
                stats_text += f"Max: {np.max(strength_values):.1f} MPa"
                self.history_stats.config(text=stats_text)
    
    def _save_history(self):
        """Save history to file"""
        try:
            history_dir = os.path.dirname(self.data_path)
            if history_dir:
                history_file = os.path.join(history_dir, "strength_prediction_history.json")
                with open(history_file, 'w') as f:
                    json.dump(self.prediction_history[-100:], f, indent=2, default=str)
        except Exception as e:
            logger.warning(f"Failed to save history: {e}")
    
    def _load_history(self):
        """Load history from file"""
        try:
            history_dir = os.path.dirname(self.data_path)
            if history_dir:
                history_file = os.path.join(history_dir, "strength_prediction_history.json")
                if os.path.exists(history_file):
                    with open(history_file, 'r') as f:
                        self.prediction_history = json.load(f)
                    self._update_history_display()
        except Exception as e:
            logger.warning(f"Failed to load history: {e}")
    
    def _toggle_fullscreen(self):
        """Toggle fullscreen mode"""
        self.fullscreen = not self.fullscreen
        self.root.attributes("-fullscreen", self.fullscreen)
        self._update_status(f"Fullscreen: {'ON' if self.fullscreen else 'OFF'}")
    
    def _change_theme(self, theme_name: str):
        """Change application theme"""
        self.theme = Theme(theme_name)
        self.colors = ColorPalette(self.theme)
        self.root.destroy()
        self.__init__(self.data_path, self.theme, self.fullscreen)
        self.run()
    
    def _update_status(self, message: str):
        """Update status bar"""
        if hasattr(self, 'status_label') and self.status_label:
            self.status_label.config(text=message)
            self.root.after(3000, lambda: self.status_label.config(text="✅ System Ready | CatBoost + MOWCA Optimized"))
    
    def run(self):
        """Run the application"""
        if not self.init_success:
            print("❌ Application failed to initialize")
            print("Check the console for error messages")
            return
        
        self.root.mainloop()


if __name__ == "__main__":
    try:
        # Update this path to your data file
        data_path = r"D:\2026 Work\My Papers\SCM-based concrete\Modelling of the data\Data\Data.csv"
        
        if not os.path.exists(data_path):
            print(f"❌ Data file not found: {data_path}")
            print("Please update the data_path variable with the correct path.")
            input("Press Enter to exit...")
            exit(1)
        
        print("🚀 Starting Concrete Strength Predictor...")
        print(f"📂 Loading data from: {data_path}")
        
        # Create and run application with PROFESSIONAL theme
        app = ConcreteStrengthPredictor(data_path, theme=Theme.PROFESSIONAL, fullscreen=True)
        app.run()
        
    except Exception as e:
        logger.error(f"Application error: {e}")
        logger.error(traceback.format_exc())
        print(f"\n❌ Fatal Error: {e}")
        print("Please check that all required packages are installed:")
        print("pip install pandas numpy matplotlib scikit-learn catboost")
        input("\nPress Enter to exit...")

2026-06-10 11:56:46,578 - INFO - Loading data from: D:\2026 Work\My Papers\SCM-based concrete\Modelling of the data\Data\Data.csv


🚀 Starting Concrete Strength Predictor...
📂 Loading data from: D:\2026 Work\My Papers\SCM-based concrete\Modelling of the data\Data\Data.csv


2026-06-10 11:56:47,126 - INFO - Target column: 'Cylinder compressive strength (MPa)'
2026-06-10 11:56:47,141 - INFO - Features (8): ['Cement(kg/m3)', 'Water(kg/m3)', 'Coarse aggregate(kg/m3)', 'Fine aggregate(kg/m3)', 'FA (kg/m3)', 'SF (kg/m3)', 'GGBFS (kg/m3)', 'SP (kg/m3)']...
2026-06-10 11:56:47,182 - INFO - Training samples: 931, Validation: 233, Testing: 292
2026-06-10 11:56:47,184 - INFO - Target range: 6.09 - 122.58 MPa
2026-06-10 11:56:47,185 - INFO - Starting model training with MOWCA hyperparameter optimization...
2026-06-10 11:56:47,189 - INFO - Starting MOWCA hyperparameter optimization...
2026-06-10 11:59:48,675 - INFO - MOWCA Iteration 1/30, Best Fitness: 6.973007
2026-06-10 12:01:18,514 - INFO - MOWCA Iteration 2/30, Best Fitness: 6.909368
2026-06-10 12:02:33,300 - INFO - MOWCA Iteration 3/30, Best Fitness: 6.903942
2026-06-10 12:03:51,593 - INFO - MOWCA Iteration 4/30, Best Fitness: 6.903942
2026-06-10 12:05:06,362 - INFO - MOWCA Iteration 5/30, Best Fitness: 6.857468
